In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import warnings
import os
warnings.filterwarnings('ignore')

print("=" * 80)
print("СРАВНЕНИЕ RANDOM FOREST И LIGHTGBM С ЛОГИРОВАНИЕМ В MLFLOW")
print("=" * 80)

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net" # ваш код здесь
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv('AWS_ACCESS_KEY_ID') # ваш код здесь
os.environ['AWS_SECRET_ACCESS_KEY'] = os.getenv('AWS_SECRET_ACCESS_KEY')
mlflow.set_registry_uri(f"s3://{os.getenv('S3_BUCKET_NAME')}")


СРАВНЕНИЕ RANDOM FOREST И LIGHTGBM С ЛОГИРОВАНИЕМ В MLFLOW


In [2]:
df = pd.read_csv("train_ver2.csv")

In [6]:
pd.set_option('display.max_columns', None)

In [7]:
df.head()

,fecha_dato,ncodpers,ind_empleado,pais_residencia,sexo,age,fecha_alta,ind_nuevo,antiguedad,indrel,ult_fec_cli_1t,indrel_1mes,tiprel_1mes,indresi,indext,conyuemp,canal_entrada,indfall,tipodom,cod_prov,nomprov,ind_actividad_cliente,renta,segmento,ind_ahor_fin_ult1,ind_aval_fin_ult1,ind_cco_fin_ult1,ind_cder_fin_ult1,ind_cno_fin_ult1,ind_ctju_fin_ult1,ind_ctma_fin_ult1,ind_ctop_fin_ult1,ind_ctpp_fin_ult1,ind_deco_fin_ult1,ind_deme_fin_ult1,ind_dela_fin_ult1,ind_ecue_fin_ult1,ind_fond_fin_ult1,ind_hip_fin_ult1,ind_plan_fin_ult1,ind_pres_fin_ult1,ind_reca_fin_ult1,ind_tjcr_fin_ult1,ind_valo_fin_ult1,ind_viv_fin_ult1,ind_nomina_ult1,ind_nom_pens_ult1,ind_recibo_ult1
0,2015-01-28,1375586,N,ES,H,35,2015-01-12,0.0,6,1.0,NaN,1.0,A,S,N,NaN,KHL,N,1.0,29.0,MALAGA,1.0,87218.10,02 - PARTICULARES,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0
1,2015-01-28,1050611,N,ES,V,23,2012-08-10,0.0,35,1.0,NaN,1.0,I,S,S,NaN,KHE,N,1.0,13.0,CIUDAD REAL,0.0,35548.74,03 - UNIVERSITARIO,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0
2,2015-01-28,1050612,N,ES,V,23,2012-08-10,0.0,35,1.0,NaN,1.0,I,S,N,NaN,KHE,N,1.0,13.0,CIUDAD REAL,0.0,122179.11,03 - UNIVERSITARIO,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0
3,2015-01-28,1050613,N,ES,H,22,2012-08-10,0.0,35,1.0,NaN,1.0,I,S,N,NaN,KHD,N,1.0,50.0,ZARAGOZA,0.0,119775.54,03 - UNIVERSITARIO,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0
4,2015-01-28,1050614,N,ES,V,23,2012-08-10,0.0,35,1.0,NaN,1.0,A,S,N,NaN,KHE,N,1.0,50.0,ZARAGOZA,1.0,NaN,03 - UNIVERSITARIO,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0


In [ ]:

# ============================================================================
# 1. ПОДГОТОВКА ДАННЫХ И МЕТРИК
# ============================================================================

def load_data():
    """Загрузка и подготовка данных"""
    print("Загрузка данных...")
    
    try:
        df = pd.read_csv('train_ver2.csv', low_memory=False)
        
        unique_clients = df['ncodpers'].unique()
        sample_clients = np.random.choice(unique_clients, size=50000, replace=False)
        df = df[df['ncodpers'].isin(sample_clients)]
        
        df['fecha_dato'] = pd.to_datetime(df['fecha_dato'])
        df['age'] = pd.to_numeric(df['age'].astype(str).str.strip(), errors='coerce')
        df['age'] = df['age'].fillna(df['age'].median())
        df['antiguedad'] = pd.to_numeric(df['antiguedad'].astype(str).str.strip(), errors='coerce')
        df['antiguedad'] = df['antiguedad'].fillna(0)
        df['renta'] = df['renta'].fillna(df['renta'].median())
        
        df['month'] = df['fecha_dato'].dt.month
        
        product_cols = [col for col in df.columns if col.startswith('ind_') and 
                       (col.endswith('_ult1') or col.endswith('_fin_ult1'))]
        
        feature_cols = ['age', 'antiguedad', 'renta', 'ind_nuevo', 'indrel', 
                       'cod_prov', 'ind_actividad_cliente', 'month']
        
        for col in product_cols:
            df[col] = df[col].fillna(0).astype(int)
        
        df['year_month'] = df['fecha_dato'].dt.to_period('M')
        unique_months = sorted(df['year_month'].unique())
        
        test_month = unique_months[-1]
        train_months = unique_months[:-1]
        
        train_df = df[df['year_month'].isin(train_months)]
        test_df = df[df['year_month'] == test_month]
        
        X_train = train_df[feature_cols].fillna(0)
        X_test = test_df[feature_cols].fillna(0)
        y_train = train_df[product_cols]
        y_test = test_df[product_cols]
        
        print(f"Данные загружены: train={X_train.shape}, test={X_test.shape}")
        print(f"Количество продуктов: {len(product_cols)}")
        
        return X_train, X_test, y_train, y_test, feature_cols, product_cols
        
    except Exception as e:
        print(f"Ошибка загрузки данных: {e}")
        print("Создание синтетических данных для демонстрации...")
        
        n_samples = 5000
        n_features = 8
        n_products = 24
        
        np.random.seed(42)
        X = np.random.randn(n_samples, n_features)
        y = np.random.randint(0, 2, size=(n_samples, n_products))
        
        X_train, X_test = X[:4000], X[4000:]
        y_train, y_test = y[:4000], y[4000:]
        
        feature_cols = [f'feature_{i}' for i in range(n_features)]
        product_cols = [f'product_{i}' for i in range(n_products)]
        
        print(f"Созданы синтетические данные: train={X_train.shape}, test={X_test.shape}")
        return X_train, X_test, y_train, y_test, feature_cols, product_cols

def calculate_metrics(y_true, y_pred_topk, k=5):
    """Вычисление метрик для рекомендательной системы"""
    precision_scores = []
    recall_scores = []
    ap_scores = []
    
    for i in range(len(y_true)):
        true_positives = len(set(y_pred_topk[i]) & set(np.where(y_true.iloc[i] == 1)[0]))
        precision = true_positives / k
        precision_scores.append(precision)
        
        relevant_items = set(np.where(y_true.iloc[i] == 1)[0])
        if len(relevant_items) > 0:
            recall = true_positives / len(relevant_items)
            recall_scores.append(recall)
        else:
            recall_scores.append(0)
        
        if len(relevant_items) > 0:
            precision_at_k = []
            num_hits = 0
            for rank, item in enumerate(y_pred_topk[i]):
                if item in relevant_items:
                    num_hits += 1
                    precision_at_k.append(num_hits / (rank + 1))
            ap = np.mean(precision_at_k) if precision_at_k else 0
        else:
            ap = 0
        ap_scores.append(ap)
    
    return {
        'precision_at_5': np.mean(precision_scores),
        'recall_at_5': np.mean(recall_scores),
        'map_at_5': np.mean(ap_scores)
    }


In [ ]:

# ============================================================================
# 2. ОБУЧЕНИЕ МОДЕЛЕЙ
# ============================================================================

def train_random_forest(X_train, y_train, X_test, y_test):
    """Обучение Random Forest модели"""
    print("\nОбучение Random Forest модели...")
    
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.multioutput import MultiOutputClassifier
    
    rf = RandomForestClassifier(
        n_estimators=30,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    
    multi_rf = MultiOutputClassifier(rf)
    multi_rf.fit(X_train, y_train)
    
    # Предсказание
    proba = multi_rf.predict_proba(X_test)
    n_samples = X_test.shape[0]
    n_products = y_test.shape[1]
    
    proba_matrix = np.zeros((n_samples, n_products))
    for i in range(n_products):
        if proba[i].shape[1] > 1:
            proba_matrix[:, i] = proba[i][:, 1]
        else:
            proba_matrix[:, i] = proba[i][:, 0]
    
    top_5_indices = np.argsort(proba_matrix, axis=1)[:, -5:][:, ::-1]
    
    metrics = calculate_metrics(y_test, top_5_indices)
    print(f"Random Forest метрики: Precision@5={metrics['precision_at_5']:.4f}, "
          f"Recall@5={metrics['recall_at_5']:.4f}, MAP@5={metrics['map_at_5']:.4f}")
    
    return {
        'model': multi_rf,
        'metrics': metrics,
        'top_k_predictions': top_5_indices
    }

def train_lightgbm(X_train, y_train, X_test, y_test):
    """Обучение LightGBM модели"""
    print("\nОбучение LightGBM модели...")
    
    try:
        import lightgbm as lgb
        from sklearn.multioutput import MultiOutputClassifier
        
        lgb_clf = lgb.LGBMClassifier(
            n_estimators=50,
            learning_rate=0.05,
            num_leaves=31,
            objective='binary',
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )
        
        multi_lgb = MultiOutputClassifier(lgb_clf)
        multi_lgb.fit(X_train, y_train)
        
        proba = multi_lgb.predict_proba(X_test)
        n_samples = X_test.shape[0]
        n_products = y_test.shape[1]
        
        proba_matrix = np.zeros((n_samples, n_products))
        for i in range(n_products):
            if proba[i].shape[1] > 1:
                proba_matrix[:, i] = proba[i][:, 1]
            else:
                proba_matrix[:, i] = proba[i][:, 0]
        
        top_5_indices = np.argsort(proba_matrix, axis=1)[:, -5:][:, ::-1]
        
        metrics = calculate_metrics(y_test, top_5_indices)
        print(f"LightGBM метрики: Precision@5={metrics['precision_at_5']:.4f}, "
              f"Recall@5={metrics['recall_at_5']:.4f}, MAP@5={metrics['map_at_5']:.4f}")
        
        return {
            'model': multi_lgb,
            'metrics': metrics,
            'top_k_predictions': top_5_indices
        }
    
    except ImportError:
        print("LightGBM не установлен. Используем GradientBoostingClassifier...")
        from sklearn.ensemble import GradientBoostingClassifier
        from sklearn.multioutput import MultiOutputClassifier
        
        gb = GradientBoostingClassifier(
            n_estimators=30,
            learning_rate=0.05,
            max_depth=5,
            random_state=42
        )
        
        multi_gb = MultiOutputClassifier(gb)
        multi_gb.fit(X_train, y_train)
        
        proba = multi_gb.predict_proba(X_test)
        n_samples = X_test.shape[0]
        n_products = y_test.shape[1]
        
        proba_matrix = np.zeros((n_samples, n_products))
        for i in range(n_products):
            if proba[i].shape[1] > 1:
                proba_matrix[:, i] = proba[i][:, 1]
            else:
                proba_matrix[:, i] = proba[i][:, 0]
        
        top_5_indices = np.argsort(proba_matrix, axis=1)[:, -5:][:, ::-1]
        
        metrics = calculate_metrics(y_test, top_5_indices)
        print(f"GradientBoosting метрики: Precision@5={metrics['precision_at_5']:.4f}, "
              f"Recall@5={metrics['recall_at_5']:.4f}, MAP@5={metrics['map_at_5']:.4f}")
        
        return {
            'model': multi_gb,
            'metrics': metrics,
            'top_k_predictions': top_5_indices
        }


In [ ]:

# ============================================================================
# 3. ЛОГИРОВАНИЕ В MLFLOW (ЛОКАЛЬНЫЙ РЕЖИМ)
# ============================================================================

def setup_mlflow_local():
    """Настройка MLflow в локальном режиме (без сервера)"""
    print("\nНастройка MLflow в локальном режиме...")
    
    mlflow.set_tracking_uri("http://localhost:5000/")
    
    experiment_name = "Bank_Product_Recommendations"
    
    try:
        experiment = mlflow.get_experiment_by_name(experiment_name)
        if experiment is None:
            experiment_id = mlflow.create_experiment(experiment_name)
            print(f"✓ Создан новый эксперимент: {experiment_name} (ID: {experiment_id})")
        else:
            experiment_id = experiment.experiment_id
            print(f"✓ Используем существующий эксперимент: {experiment_name} (ID: {experiment_id})")
    except Exception as e:
        try:
            experiment_id = mlflow.create_experiment(experiment_name)
            print(f"✓ Создан эксперимент: {experiment_name} (ID: {experiment_id})")
        except:
            experiment = mlflow.get_experiment_by_name(experiment_name)
            experiment_id = experiment.experiment_id
            print(f"✓ Используем существующий эксперимент: {experiment_name} (ID: {experiment_id})")
    
    return experiment_id

def log_model_to_mlflow(model_name, model_info, experiment_id):
    """Логирование одной модели в MLflow"""
    print(f"\nЛогирование модели {model_name} в MLflow...")
    
    with mlflow.start_run(run_name=f"{model_name}_model", experiment_id=experiment_id) as run:
        if model_name == "random_forest":
            mlflow.log_params({
                'n_estimators': 30,
                'max_depth': 10,
                'model_type': 'RandomForest_MultiOutput'
            })
        elif model_name == "lightgbm":
            mlflow.log_params({
                'n_estimators': 50,
                'learning_rate': 0.05,
                'num_leaves': 31,
                'model_type': 'LightGBM_MultiOutput'
            })
        

        mlflow.log_metrics(model_info['metrics'])
        
        model_info_obj = mlflow.sklearn.log_model(
            sk_model=model_info['model'],
        )
        
        print(f"✓ Модель {model_name} залогирована")
        print(f"  Run ID: {run.info.run_id}")
        print(f"  Метрики: {model_info['metrics']}")
        
        return run.info.run_id


In [ ]:

# ============================================================================
# 4. ОСНОВНОЙ ПРОЦЕСС
# ============================================================================

def main():
    
    X_train, X_test, y_train, y_test, feature_cols, product_cols = load_data()
    
    experiment_id = setup_mlflow_local()
    
    print("\n" + "="*60)
    print("ОБУЧЕНИЕ МОДЕЛЕЙ")
    print("="*60)
    
    rf_result = train_random_forest(X_train, y_train, X_test, y_test)
    
    lgb_result = train_lightgbm(X_train, y_train, X_test, y_test)
    
    print("\n" + "="*60)
    print("ЛОГИРОВАНИЕ В MLFLOW")
    print("="*60)
    
    rf_run_id = log_model_to_mlflow("random_forest", rf_result, experiment_id)
    lgb_run_id = log_model_to_mlflow("lightgbm", lgb_result, experiment_id)
    
    print("\n" + "="*60)
    print("СРАВНЕНИЕ РЕЗУЛЬТАТОВ")
    print("="*60)
    
    comparison_data = []
    for model_name, result in [("Random Forest", rf_result), ("LightGBM", lgb_result)]:
        row = {"Model": model_name}
        row.update(result['metrics'])
        comparison_data.append(row)
    
    comparison_df = pd.DataFrame(comparison_data)
    
    display_df = comparison_df.copy()
    display_df.columns = ["Model", "Precision@5", "Recall@5", "MAP@5"]
    
    print("\nСравнительная таблица:")
    print(display_df.to_string(index=False))
    
    best_idx = comparison_df['map_at_5'].idxmax()
    best_model = comparison_df.loc[best_idx]
    
    best_model_display = {
        "Model": best_model["Model"],
        "Precision@5": best_model["precision_at_5"],
        "Recall@5": best_model["recall_at_5"],
        "MAP@5": best_model["map_at_5"]
    }
    
    print(f"\n✓ Лучшая модель: {best_model_display['Model']}")
    print(f"  MAP@5: {best_model_display['MAP@5']:.4f}")
    print(f"  Precision@5: {best_model_display['Precision@5']:.4f}")
    print(f"  Recall@5: {best_model_display['Recall@5']:.4f}")
    
    comparison_df.to_csv('rf_vs_lgb_comparison.csv', index=False)
    print("\n✓ Результаты сравнения сохранены в 'rf_vs_lgb_comparison.csv'")
    
    if best_model_display['Model'] == "Random Forest":
        import joblib
        joblib.dump(rf_result['model'], 'best_model.bin')
        print("✓ Лучшая модель (Random Forest) сохранена в 'best_model.bin'")
    else:
        import joblib
        joblib.dump(lgb_result['model'], 'best_model.bin')
        print("✓ Лучшая модель (LightGBM) сохранена в 'best_model.bin'")
    
    return rf_result, lgb_result, comparison_df


In [ ]:

# ============================================================================
# ЗАПУСК
# ============================================================================

if __name__ == "__main__":
    try:
        print("\n" + "="*80)
        print("ЗАПУСК ЭКСПЕРИМЕНТА")
        print("="*80)
        
        rf_result, lgb_result, comparison_df = main()
        
        print("\n" + "="*80)
        print("ЭКСПЕРИМЕНТ УСПЕШНО ЗАВЕРШЕН!")
        print("="*80)
        
    except Exception as e:
        print(f"\n Ошибка: {e}")
        import traceback
        traceback.print_exc()